# Porównanie trzech metod wyszukiwania

Korpus: 40 artykułów z polskiej Wikipedii (`sample_docs/`). Wymaga zbudowanej bazy:
`python scripts/build_index.py`.

Zapytania są trzech rodzajów:
1. **dosłowne** - słowo występuje w tekście w tej samej formie,
2. **parafrazy** - opis tematu innymi słowami niż w artykule,
3. **odmiana** - słowo jest w tekście, ale w innej formie gramatycznej,
4. **wieloznaczne** - jedno słowo, kilka możliwych tematów.

Dla każdego zapytania sam oznaczyłem, które dokumenty uważam za trafne
(to moja ocena, nie dane z zewnątrz). Na tej podstawie liczę MRR
(średnia odwrotność pozycji pierwszego trafnego dokumentu: 1 = trafny na 1. miejscu,
0,5 = na 2., 0 = brak w top-k).

In [1]:
import os, sys
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, ".")
import pandas as pd
from document_store import DocumentStore
from search_engine import SearchEngine
from ranking_compare import pairwise_stats, positions_table
pd.set_option("display.max_colwidth", 60)

store = DocumentStore("storage")
engine = SearchEngine(store)
names = {d.id: d.name for d in store.get_all()}
print("dokumentów w bazie:", len(names))
K = 5

dokumentów w bazie: 40


In [2]:
QUERIES = [
    ("dosłowne", "fotosynteza", {"Fotosynteza.txt"}),
    ("dosłowne", "NATO", {"NATO.txt"}),
    ("dosłowne", "skoki narciarskie", {"Adam_Małysz.txt"}),
    ("parafraza", "zryw przeciw Niemcom w stolicy w 1944", {"Powstanie_warszawskie.txt"}),
    ("parafraza", "maszyny, które same się uczą", {"Uczenie_maszynowe.txt", "Sztuczna_inteligencja.txt"}),
    ("parafraza", "epopeja narodowa o szlachcie na Litwie", {"Pan_Tadeusz.pdf", "Adam_Mickiewicz.txt"}),
    ("parafraza", "najwyższe góry w Polsce", {"Tatry.pdf"}),
    ("parafraza", "pisarz science fiction", {"Stanisław_Lem.txt"}),
    ("parafraza", "sojusz wojskowy państw zachodnich", {"NATO.txt"}),
    ("odmiana", "Krzyżacy", {"Bitwa_pod_Grunwaldem.txt"}),
    ("odmiana", "polska noblistka", {"Maria_Skłodowska_Curie.txt", "Wisława_Szymborska.txt"}),
    ("wieloznaczne", "unia", {"Unia_Europejska.pdf", "Unia_lubelska.txt"}),
    ("wieloznaczne", "Wisła", {"Wisła.txt"}),
]

def reciprocal_rank(ranking, relevant):
    for i, name in enumerate(ranking, 1):
        if name in relevant:
            return 1 / i
    return 0.0

rows, corr_rows, all_results = [], [], {}
for kind, q, relevant in QUERIES:
    res = engine.search_all(q, K)
    all_results[q] = res
    ids = {m: [r.doc_id for r in v] for m, v in res.items()}
    for m, v in res.items():
        rows.append({"rodzaj": kind, "zapytanie": q, "metoda": m,
                     "RR": reciprocal_rank([r.doc_name for r in v], relevant),
                     "top-3": ", ".join(r.doc_name.rsplit(".", 1)[0] for r in v[:3])})
    st = pairwise_stats(ids, K)
    st.insert(0, "zapytanie", q)
    corr_rows.append(st)
results = pd.DataFrame(rows)
corr = pd.concat(corr_rows, ignore_index=True)
results.pivot(index="zapytanie", columns="metoda", values="top-3")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

metoda,Semantyczne,Słowa kluczowe,TF-IDF
zapytanie,,,
Krzyżacy,"Bitwa_pod_Grunwaldem, Pan_Tadeusz, Chrzest_Polski",Bitwa_pod_Grunwaldem,
NATO,"NATO, Unia_Europejska, Morze_Bałtyckie","NATO, Kraków","NATO, Kraków"
Wisła,"Wisła, Warszawa, Kraków","Wisła, Adam_Małysz, Morze_Bałtyckie","Wisła, Kraków, Warszawa"
epopeja narodowa o szlachcie na Litwie,"Pan_Tadeusz, Adam_Mickiewicz, Chrzest_Polski","Pan_Tadeusz, Unia_lubelska, Konstytucja_3_maja","Pan_Tadeusz, Unia_lubelska, Adam_Mickiewicz"
fotosynteza,"Fotosynteza, Kwas_deoksyrybonukleinowy, Morze_Bałtyckie",Fotosynteza,Fotosynteza
"maszyny, które same się uczą","Uczenie_maszynowe, Sztuczna_inteligencja, Linux","Sztuczna_inteligencja, Chrzest_Polski, Linux","Uczenie_maszynowe, Sztuczna_inteligencja, Politechnika_K..."
najwyższe góry w Polsce,"Tatry, Wisła, Kraków","Tatry, Kraków, Konstytucja_3_maja","Tatry, Wisła, Tour_de_Pologne"
pisarz science fiction,"Stanisław_Lem, Maria_Skłodowska_Curie, Python","Stanisław_Lem, Adam_Mickiewicz, Andrzej_Wajda","Stanisław_Lem, Wisława_Szymborska, Andrzej_Wajda"
polska noblistka,"Wisława_Szymborska, Maria_Skłodowska_Curie, Chrzest_Polski","Maria_Skłodowska_Curie, Bitwa_pod_Grunwaldem, Konstytucj...","Wisława_Szymborska, Maria_Skłodowska_Curie, Unia_lubelska"


## Przykład: pełna tabela pozycji dla jednego zapytania

In [3]:
q = "zryw przeciw Niemcom w stolicy w 1944"
ids = {m: [r.doc_id for r in v] for m, v in all_results[q].items()}
display(positions_table(ids, names, K))
display(pairwise_stats(ids, K).round(3))

,dokument,Słowa kluczowe,TF-IDF,Semantyczne,rozrzut
0,Powstanie_warszawskie.txt,1.0,1.0,2.0,1
1,Warszawa.txt,2.0,2.0,1.0,1
2,Fryderyk_Chopin.txt,3.0,NaN,NaN,3
3,Kraków.txt,NaN,3.0,NaN,3
4,Chrzest_Polski.txt,NaN,NaN,3.0,3
5,Bitwa_pod_Grunwaldem.txt,4.0,5.0,NaN,2
6,Letnie_Igrzyska_Olimpijskie_2024.txt,NaN,4.0,NaN,2
7,Niezależny_Samorządny_Związek_Zawodowy_Solidarność.txt,NaN,NaN,4.0,2
8,Pan_Tadeusz.pdf,5.0,NaN,NaN,1
9,NATO.txt,NaN,NaN,5.0,1


,para metod,Spearman,Kendall tau,Jaccard@5,wspólne w top-k
0,Słowa kluczowe - TF-IDF,0.476,0.400,0.429,3
1,Słowa kluczowe - Semantyczne,0.297,0.171,0.250,2
2,TF-IDF - Semantyczne,0.297,0.171,0.250,2


## Trafność (MRR) według rodzaju zapytania

In [4]:
mrr = results.pivot_table(index="rodzaj", columns="metoda", values="RR", aggfunc="mean")
mrr.loc["średnio"] = results.groupby("metoda").RR.mean()
mrr.round(3)

metoda,Semantyczne,Słowa kluczowe,TF-IDF
rodzaj,,,
dosłowne,1.000,1.0,1.000
odmiana,1.000,1.0,0.500
parafraza,0.917,1.0,1.000
wieloznaczne,1.000,1.0,1.000
średnio,0.962,1.0,0.923


In [5]:
results.pivot(index="zapytanie", columns="metoda", values="RR").round(2)

metoda,Semantyczne,Słowa kluczowe,TF-IDF
zapytanie,,,
Krzyżacy,1.0,1.0,0.0
NATO,1.0,1.0,1.0
Wisła,1.0,1.0,1.0
epopeja narodowa o szlachcie na Litwie,1.0,1.0,1.0
fotosynteza,1.0,1.0,1.0
"maszyny, które same się uczą",1.0,1.0,1.0
najwyższe góry w Polsce,1.0,1.0,1.0
pisarz science fiction,1.0,1.0,1.0
polska noblistka,1.0,1.0,1.0


## Jak bardzo różnią się rankingi? (średnio po zapytaniach)

In [6]:
mean_corr = corr.groupby("para metod")[["Spearman", "Kendall tau", f"Jaccard@{K}"]].mean()
# zapis do results/ - z tych plików powstają wykresy do prezentacji (scripts/make_figures.py)
Path("results").mkdir(exist_ok=True)
results.to_csv("results/zapytania_rr.csv", index=False)
mean_corr.round(4).to_csv("results/korelacje_srednie.csv")
mean_corr.round(3)

,Spearman,Kendall tau,Jaccard@5
para metod,,,
Słowa kluczowe - Semantyczne,0.321,0.263,0.282
Słowa kluczowe - TF-IDF,0.519,0.469,0.518
TF-IDF - Semantyczne,0.428,0.337,0.336


## Wnioski

In [7]:
for kind in list(mrr.index):
    row = mrr.loc[kind]
    winners = ", ".join(row[row == row.max()].index)
    print(f"{kind:13s} najwyższe MRR {row.max():.2f}: {winners}")
pairs = corr.groupby("para metod")[f"Jaccard@{K}"].mean()
print("Najbardziej podobne rankingi:", pairs.idxmax(), " najmniej:", pairs.idxmin())

dosłowne      najwyższe MRR 1.00: Semantyczne, Słowa kluczowe, TF-IDF
odmiana       najwyższe MRR 1.00: Semantyczne, Słowa kluczowe
parafraza     najwyższe MRR 1.00: Słowa kluczowe, TF-IDF
wieloznaczne  najwyższe MRR 1.00: Semantyczne, Słowa kluczowe, TF-IDF
średnio       najwyższe MRR 1.00: Słowa kluczowe
Najbardziej podobne rankingi: Słowa kluczowe - TF-IDF  najmniej: Słowa kluczowe - Semantyczne


- Na tym korpusie (40 artykułów o wyraźnie różnych tematach) wszystkie trzy metody
  zwykle stawiają trafny dokument na 1. miejscu. Artykuły Wikipedii są długie i zawierają
  wiele słów z zapytania, więc nawet proste słowa kluczowe często trafiają.
- TF-IDF przegrywa na zapytaniu "Krzyżacy": lematyzator spaCy zamienia to słowo na błędny
  lemat, więc zapytanie nie ma wspólnego słowa z dokumentem. Słowa kluczowe znajdują
  dokładną formę, a model semantyczny - znaczenie.
- Wyszukiwanie semantyczne przy parafrazie "zryw przeciw Niemcom w stolicy w 1944" daje
  trafny dokument na 2. miejscu (na 1. jest bliski tematycznie artykuł o Warszawie).
  Ta metoda zawsze zwraca k wyników - nie ma progu "nic nie pasuje".
- Metody zgadzają się co do najlepszego dokumentu, ale dalsza kolejność bardzo się różni
  (niskie Jaccard@k i korelacje rang w tabeli wyżej). Najbardziej podobne są słowa
  kluczowe i TF-IDF (obie oparte na słowach), najmniej - słowa kluczowe i semantyczne.
- Kiedy która metoda: słowa kluczowe - gdy znamy dokładną nazwę lub frazę (cudzysłów);
  TF-IDF - gdy słowa występują w różnych formach i ważne są słowa rzadkie;
  semantyczne - gdy opisujemy temat własnymi słowami.